# Partie 3 : Spark SQL et DataFrames
Refaire et enrichir l'analyse de la partie 2 avec l'API structurée (DataFrame + SQL).

In [1]:
import time
from common import get_spark, LAKE
spark = get_spark("SahlPay-P3-DataFrames")
sc = spark.sparkContext

# attendre que les 2 executors soient enregistrés (sinon defaultParallelism = 2)
for _ in range(30):
    if sc._jsc.sc().getExecutorMemoryStatus().size() >= 3:   # driver + 2 executors
        break
    time.sleep(1)
print("Spark", spark.version, "| master:", sc.master, "| cœurs:", sc.defaultParallelism)

https://repos.spark-packages.org added as a remote repository with the name: repo-1
Ivy Default Cache set to: /tmp/ivy/cache
The jars for the packages stored in: /tmp/ivy/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-364615a3-c056-42bb-99a4-4451ebaf22b7;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central


	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central


	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central


:: resolution report :: resolve 838ms :: artifacts dl 16ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	commons-logging#commons-logging;1.1.3 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.commons#commons-pool2;2.11.1 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.3.4 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.3.4 from central in [default]
	org.apache.kafka#kafka-clients;3.4.1 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 from central in [default]
	org.lz4#lz4-java;1.8.0 from central in [default]


26/11/24 17:02:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.1 | master: spark://spark-master:7077 | cœurs: 4


## Q1 : lecture avec schéma explicite (sans inférence)

In [2]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

tx_schema = StructType([
    StructField("tx_id", StringType(), False),
    StructField("type", StringType()),
    StructField("payer_wallet", StringType()),
    StructField("payee_id", StringType()),
    StructField("device_id", StringType()),
    StructField("city", StringType()),
    StructField("channel", StringType()),
    StructField("status", StringType()),
    StructField("amount_mad", DoubleType()),
    StructField("event_time", TimestampType()),
])
merchant_schema = StructType([
    StructField("merchant_id", StringType(), False),
    StructField("name", StringType()),
    StructField("city", StringType()),
    StructField("category", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
])

tx = spark.read.schema(tx_schema).json(f"{LAKE}/bronze/history/transactions_history.jsonl")
merchants = spark.read.schema(merchant_schema).json(f"{LAKE}/ref/merchants/")
tx.printSchema()
merchants.show(5, truncate=False)
# Le producteur injecte ~2 % de doublons (même tx_id + status) : on les retire
completed = tx.filter("status = 'COMPLETED'").dropDuplicates(["tx_id", "status"]).cache()

26/11/24 17:02:38 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- tx_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- payer_wallet: string (nullable = true)
 |-- payee_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount_mad: double (nullable = true)
 |-- event_time: timestamp (nullable = true)



+-----------+-----------------------+----------+---------+--------+---------+
|merchant_id|name                   |city      |category |latitude|longitude|
+-----------+-----------------------+----------+---------+--------+---------+
|M-001      |Epicerie Tanger 001    |Tanger    |epicerie |35.78366|-5.85951 |
|M-002      |Station Casablanca 002 |Casablanca|carburant|33.59077|-7.55058 |
|M-003      |Boutique Casablanca 003|Casablanca|vetements|33.52628|-7.63043 |
|M-004      |Epicerie Casablanca 004|Casablanca|epicerie |33.57922|-7.5682  |
|M-005      |Boutique Fes 005       |Fes       |vetements|33.99014|-4.99887 |
+-----------+-----------------------+----------+---------+--------+---------+
only showing top 5 rows



## Q2 : montant par ville, API DataFrame puis SQL (à comparer avec la partie 2)

In [3]:
t0 = time.time()
by_city = (completed.groupBy("city")
           .agg(F.round(F.sum("amount_mad"), 2).alias("total_mad"),
                F.count("*").alias("nb"),
                F.round(F.avg("amount_mad"), 2).alias("moyenne"))
           .orderBy(F.desc("total_mad")))
by_city.show()
t_df = time.time() - t0
print("Temps DataFrame (lecture S3 + cache inclus) :", round(t_df, 2), "s")

+----------+----------+----+-------+
|      city| total_mad|  nb|moyenne|
+----------+----------+----+-------+
|Casablanca|1771958.41|4044| 438.17|
|     Rabat| 905406.01|2123| 426.47|
| Marrakech|  812544.5|1421| 571.81|
|    Tanger| 661911.48|1866| 354.72|
|       Fes| 496963.56|1365| 364.08|
|     Oujda| 326583.26| 419| 779.43|
|   Tetouan| 315692.32| 399| 791.21|
|    Meknes| 307520.92| 665| 462.44|
|    Agadir| 293650.67| 919| 319.53|
|   Kenitra| 274320.34| 807| 339.93|
+----------+----------+----+-------+

Temps DataFrame (lecture S3 + cache inclus) : 9.07 s


In [4]:
completed.createOrReplaceTempView("completed")
t0 = time.time()
spark.sql("""SELECT city, ROUND(SUM(amount_mad),2) AS total_mad, COUNT(*) AS nb,
                    ROUND(AVG(amount_mad),2) AS moyenne
             FROM completed GROUP BY city ORDER BY total_mad DESC""").show()
t_sql = time.time() - t0
print("Temps SQL (données déjà en cache) :", round(t_sql, 2), "s")

+----------+----------+----+-------+
|      city| total_mad|  nb|moyenne|
+----------+----------+----+-------+
|Casablanca|1771958.41|4044| 438.17|
|     Rabat| 905406.01|2123| 426.47|
| Marrakech|  812544.5|1421| 571.81|
|    Tanger| 661911.48|1866| 354.72|
|       Fes| 496963.56|1365| 364.08|
|     Oujda| 326583.26| 419| 779.43|
|   Tetouan| 315692.32| 399| 791.21|
|    Meknes| 307520.92| 665| 462.44|
|    Agadir| 293650.67| 919| 319.53|
|   Kenitra| 274320.34| 807| 339.93|
+----------+----------+----+-------+

Temps SQL (données déjà en cache) : 0.84 s


In [5]:
# Point de contrôle : mêmes chiffres que la partie 2 (RDD)
rdd_p2 = {"Casablanca": (1771958.41, 4044), "Rabat": (905406.01, 2123), "Marrakech": (812544.50, 1421),
          "Tanger": (661911.48, 1866), "Fes": (496963.56, 1365), "Oujda": (326583.26, 419),
          "Tetouan": (315692.32, 399), "Meknes": (307520.92, 665), "Agadir": (293650.67, 919),
          "Kenitra": (274320.34, 807)}
ok = all(abs(r.total_mad - rdd_p2[r.city][0]) < 0.01 and r.nb == rdd_p2[r.city][1] for r in by_city.collect())
print("Résultats identiques à la partie 2 :", ok)

Résultats identiques à la partie 2 : True


## Q3 : jointure P2M x merchants : montant par catégorie et par ville

In [6]:
p2m = completed.filter("type = 'P2M'")
joined = p2m.join(merchants.withColumnRenamed("city", "m_city"),
                  p2m.payee_id == merchants.merchant_id, "inner")
cat_city = (joined.groupBy("city", "category")
            .agg(F.round(F.sum("amount_mad"), 2).alias("total_mad"), F.count("*").alias("nb"))
            .orderBy("city", F.desc("total_mad")))
t0 = time.time()
cat_city.show(80, truncate=False)
t_join_bhj = time.time() - t0
print("Temps jointure (BroadcastHashJoin) :", round(t_join_bhj, 2), "s")

+----------+------------+---------+---+
|city      |category    |total_mad|nb |
+----------+------------+---------+---+
|Agadir    |epicerie    |44088.71 |230|
|Agadir    |restauration|37838.75 |152|
|Agadir    |carburant   |36706.33 |95 |
|Agadir    |pharmacie   |14215.24 |73 |
|Agadir    |vetements   |12473.68 |35 |
|Agadir    |transport   |8484.37  |25 |
|Agadir    |telecom     |5237.35  |29 |
|Casablanca|carburant   |319560.01|726|
|Casablanca|vetements   |206017.05|531|
|Casablanca|epicerie    |88485.98 |525|
|Casablanca|restauration|83247.54 |419|
|Casablanca|transport   |71485.92 |354|
|Casablanca|e-commerce  |66108.19 |120|
|Casablanca|pharmacie   |57139.75 |189|
|Casablanca|telecom     |28256.33 |149|
|Fes       |vetements   |112897.15|242|
|Fes       |e-commerce  |103866.73|166|
|Fes       |carburant   |32979.93 |80 |
|Fes       |transport   |32573.8  |155|
|Fes       |telecom     |23338.6  |171|
|Fes       |epicerie    |21239.09 |99 |
|Fes       |restauration|13385.91 |69 |


In [7]:
# Vue synthétique : montant par catégorie (toutes villes)
joined.groupBy("category").agg(F.round(F.sum("amount_mad"), 2).alias("total_mad"),
                               F.count("*").alias("nb")).orderBy(F.desc("total_mad")).show()

+------------+---------+----+
|    category|total_mad|  nb|
+------------+---------+----+
|   carburant|753053.08|1821|
|   vetements|507311.51|1262|
|  e-commerce|473578.09| 891|
|    epicerie|357186.24|2021|
|   pharmacie|306635.69|1101|
|restauration|258920.46|1166|
|   transport|211639.19| 998|
|     telecom|175108.71| 946|
+------------+---------+----+



## Q4 : UDF Python `amount_band` contre `when/otherwise` natif

In [8]:
from pyspark.sql.functions import udf

@udf(StringType())
def amount_band(a):
    if a is None:
        return None
    return "micro" if a < 50 else ("standard" if a <= 2000 else "eleve")

def band_native(c):
    return F.when(c < 50, "micro").when(c <= 2000, "standard").otherwise("eleve")

tx.cache().count()   # même point de départ en mémoire pour les deux versions
for name, col in [("UDF Python", amount_band("amount_mad")),
                  ("when/otherwise", band_native(F.col("amount_mad")))]:
    times = []
    for _ in range(3):
        t0 = time.time()
        tx.withColumn("amount_band", col).write.format("noop").mode("overwrite").save()  # force le calcul complet
        times.append(time.time() - t0)
    print(f"{name:<15} meilleur de 3 : {min(times):.2f} s   ({', '.join(f'{t:.2f}' for t in times)})")

tx.withColumn("amount_band", band_native(F.col("amount_mad"))).groupBy("amount_band").count().show()

UDF Python      meilleur de 3 : 0.42 s   (1.54, 0.42, 0.67)


when/otherwise  meilleur de 3 : 0.19 s   (0.31, 0.26, 0.19)


+-----------+-----+
|amount_band|count|
+-----------+-----+
|      eleve| 1830|
|      micro| 7275|
|   standard|40895|
+-----------+-----+



In [9]:
# Le plan montre la différence : BatchEvalPython (sortie de la JVM) vs simple CASE WHEN
tx.withColumn("amount_band", amount_band("amount_mad")).select("tx_id", "amount_band").explain()
tx.withColumn("amount_band", band_native(F.col("amount_mad"))).select("tx_id", "amount_band").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [tx_id#0, pythonUDF0#3756 AS amount_band#3642]
   +- BatchEvalPython [amount_band(amount_mad#8)#3641], [pythonUDF0#3756]
      +- InMemoryTableScan [tx_id#0, amount_mad#8]
            +- InMemoryRelation [tx_id#0, type#1, payer_wallet#2, payee_id#3, device_id#4, city#5, channel#6, status#7, amount_mad#8, event_time#9], StorageLevel(disk, memory, deserialized, 1 replicas)
                  +- FileScan json [tx_id#0,type#1,payer_wallet#2,payee_id#3,device_id#4,city#5,channel#6,status#7,amount_mad#8,event_time#9] Batched: false, DataFilters: [], Format: JSON, Location: InMemoryFileIndex(1 paths)[s3a://lake/bronze/history/transactions_history.jsonl], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<tx_id:string,type:string,payer_wallet:string,payee_id:string,device_id:string,city:string,...




== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [tx_id#0, CASE WHEN (amount_mad#8 < 50.0) THEN micro WHEN (amount_mad#8 <= 2000.0) THEN standard ELSE eleve END AS amount_band#3807]
   +- InMemoryTableScan [amount_mad#8, tx_id#0]
         +- InMemoryRelation [tx_id#0, type#1, payer_wallet#2, payee_id#3, device_id#4, city#5, channel#6, status#7, amount_mad#8, event_time#9], StorageLevel(disk, memory, deserialized, 1 replicas)
               +- FileScan json [tx_id#0,type#1,payer_wallet#2,payee_id#3,device_id#4,city#5,channel#6,status#7,amount_mad#8,event_time#9] Batched: false, DataFilters: [], Format: JSON, Location: InMemoryFileIndex(1 paths)[s3a://lake/bronze/history/transactions_history.jsonl], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<tx_id:string,type:string,payer_wallet:string,payee_id:string,device_id:string,city:string,...




## Q5 : écriture Silver en Parquet partitionné par `city`

In [10]:
silver = (tx.dropDuplicates(["tx_id", "status"])
            .withColumn("amount_band", band_native(F.col("amount_mad")))
            .withColumn("date", F.to_date("event_time")))
t0 = time.time()
silver.write.mode("overwrite").partitionBy("city").parquet(f"{LAKE}/silver/transactions_history/")
print("Écriture Silver :", round(time.time() - t0, 2), "s")

back = spark.read.parquet(f"{LAKE}/silver/transactions_history/")
print("Lignes Bronze :", tx.count(), "| lignes Silver (sans doublons) :", back.count())
back.groupBy("city").count().orderBy("city").show()

Écriture Silver : 8.38 s


Lignes Bronze : 50000 | lignes Silver (sans doublons) : 49066


+----------+-----+
|      city|count|
+----------+-----+
|    Agadir| 3243|
|Casablanca|14103|
|       Fes| 4749|
|   Kenitra| 2856|
| Marrakech| 4960|
|    Meknes| 2302|
|     Oujda| 1473|
|     Rabat| 7452|
|    Tanger| 6556|
|   Tetouan| 1372|
+----------+-----+



## Q6 : `explain(True)` de la jointure

In [11]:
joined.explain(True)

== Parsed Logical Plan ==
Join Inner, (payee_id#3 = merchant_id#20)
:- Filter (type#1 = P2M)
:  +- Deduplicate [tx_id#0, status#7]
:     +- Filter (status#7 = COMPLETED)
:        +- Relation [tx_id#0,type#1,payer_wallet#2,payee_id#3,device_id#4,city#5,channel#6,status#7,amount_mad#8,event_time#9] json
+- Project [merchant_id#20, name#21, city#22 AS m_city#1052, category#23, latitude#24, longitude#25]
   +- Relation [merchant_id#20,name#21,city#22,category#23,latitude#24,longitude#25] json

== Analyzed Logical Plan ==
tx_id: string, type: string, payer_wallet: string, payee_id: string, device_id: string, city: string, channel: string, status: string, amount_mad: double, event_time: timestamp, merchant_id: string, name: string, m_city: string, category: string, latitude: double, longitude: double
Join Inner, (payee_id#3 = merchant_id#20)
:- Filter (type#1 = P2M)
:  +- Deduplicate [tx_id#0, status#7]
:     +- Filter (status#7 = COMPLETED)
:        +- Relation [tx_id#0,type#1,payer_wallet#

### Bonus : forcer un SortMergeJoin et comparer

In [12]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
joined_smj = p2m.join(merchants.withColumnRenamed("city", "m_city"),
                      p2m.payee_id == merchants.merchant_id, "inner")
joined_smj.explain()
t0 = time.time()
(joined_smj.groupBy("city", "category").agg(F.sum("amount_mad")).write.format("noop").mode("overwrite").save())
t_smj = time.time() - t0
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)
t0 = time.time()
(joined.groupBy("city", "category").agg(F.sum("amount_mad")).write.format("noop").mode("overwrite").save())
t_bhj = time.time() - t0
print(f"SortMergeJoin : {t_smj:.2f} s | BroadcastHashJoin : {t_bhj:.2f} s")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [payee_id#3], [merchant_id#20], Inner
   :- Sort [payee_id#3 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(payee_id#3, 8), ENSURE_REQUIREMENTS, [plan_id=1318]
   :     +- Filter ((isnotnull(type#1) AND (type#1 = P2M)) AND isnotnull(payee_id#3))
   :        +- InMemoryTableScan [tx_id#0, type#1, payer_wallet#2, payee_id#3, device_id#4, city#5, channel#6, status#7, amount_mad#8, event_time#9], [isnotnull(type#1), (type#1 = P2M), isnotnull(payee_id#3)]
   :              +- InMemoryRelation [tx_id#0, type#1, payer_wallet#2, payee_id#3, device_id#4, city#5, channel#6, status#7, amount_mad#8, event_time#9], StorageLevel(disk, memory, deserialized, 1 replicas)
   :                    +- AdaptiveSparkPlan isFinalPlan=false
   :                       +- SortAggregate(key=[tx_id#0, status#7], functions=[first(type#1, false), first(payer_wallet#2, false), first(payee_id#3, false), first(device_id#4, false), f

SortMergeJoin : 1.33 s | BroadcastHashJoin : 1.31 s


## Q7 : temps mesurés (pour la comparaison RDD / DataFrame / SQL du rapport)

In [13]:
print(f"RDD (partie 2, lecture + 3 agrégations) : 5.49 s")
print(f"DataFrame (lecture + agrégation)        : {t_df:.2f} s")
print(f"SQL (sur données en cache)               : {t_sql:.2f} s")

RDD (partie 2, lecture + 3 agrégations) : 5.49 s
DataFrame (lecture + agrégation)        : 9.07 s
SQL (sur données en cache)               : 0.84 s


In [14]:
# Pause pour les captures Spark UI (sans effet si le fichier /tmp/hold_p3 n'existe pas)
import os
while os.path.exists("/tmp/hold_p3"):
    time.sleep(2)

In [15]:
spark.stop()